In [1]:
from pathlib import Path
import sys
import random

import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
SEED = 42


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(SEED)

In [3]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)

Random seed: 42


In [4]:
NUM_QUBITS = 4

BATCH_SIZE = 32
EPOCHS = 30

LEARNING_RATE = 0.01

DEVICE = torch.device("cpu")

print("=" * 50)
print("NOTEBOOK 04B — SCALED PCA VQC")
print("=" * 50)

print("Seed       :", SEED)
print("Qubits     :", NUM_QUBITS)
print("Batch size :", BATCH_SIZE)
print("Epochs     :", EPOCHS)
print("LR         :", LEARNING_RATE)

NOTEBOOK 04B — SCALED PCA VQC
Seed       : 42
Qubits     : 4
Batch size : 32
Epochs     : 30
LR         : 0.01


In [5]:
DATA_DIR = PROJECT_ROOT / "data" / "binary"

X_train = np.load(DATA_DIR / "X_train.npy")
X_test = np.load(DATA_DIR / "X_test.npy")

y_train = np.load(DATA_DIR / "y_train.npy")
y_test = np.load(DATA_DIR / "y_test.npy")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (11824, 4)
X_test : (2956, 4)
y_train: (11824,)
y_test : (2956,)


In [6]:
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train,
)

print("Training samples  :", len(X_train_split))
print("Validation samples:", len(X_val))
print("Test samples      :", len(X_test))

Training samples  : 9459
Validation samples: 2365
Test samples      : 2956


In [7]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

print("Project root:")
print(PROJECT_ROOT)

print("\nSearching for PCA files...")

for path in PROJECT_ROOT.rglob("*.npy"):
    if "pca" in path.name.lower():
        print(path)

Project root:
c:\Work\Quantum-Adversarial-Robustness

Searching for PCA files...


In [8]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train_split
)

X_val_scaled = scaler.transform(
    X_val
)

X_test_scaled = scaler.transform(
    X_test
)

print("Training mean:")
print(X_train_scaled.mean(axis=0))

print()

print("Training std:")
print(X_train_scaled.std(axis=0))

Training mean:
[ 2.69486422e-17 -3.94370373e-18  2.68077956e-17 -8.26299830e-18]

Training std:
[1. 1. 1. 1.]


In [9]:
import joblib

SCALER_DIR = PROJECT_ROOT / "results" / "preprocessing"

SCALER_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SCALER_PATH = (
    SCALER_DIR
    / "standard_scaler_04B_seed42.joblib"
)

joblib.dump(
    scaler,
    SCALER_PATH
)

print("Scaler saved:")
print(SCALER_PATH)

Scaler saved:
c:\Work\Quantum-Adversarial-Robustness\results\preprocessing\standard_scaler_04B_seed42.joblib


In [10]:
X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train_split,
    dtype=torch.float32
).reshape(-1, 1)

X_val_tensor = torch.tensor(
    X_val_scaled,
    dtype=torch.float32
)

y_val_tensor = torch.tensor(
    y_val,
    dtype=torch.float32
).reshape(-1, 1)

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.float32
).reshape(-1, 1)

print("Train:", X_train_tensor.shape)
print("Val  :", X_val_tensor.shape)
print("Test :", X_test_tensor.shape)

Train: torch.Size([9459, 4])
Val  : torch.Size([2365, 4])
Test : torch.Size([2956, 4])


In [11]:
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [12]:
from src.models.quantum_model import create_model
from src.models.hybrid_classifier import HybridClassifier
from src.training.trainer import Trainer

In [13]:
quantum_model = create_model(
    num_qubits=NUM_QUBITS
)

model = HybridClassifier(
    quantum_model=quantum_model
)

model = model.to(DEVICE)

print(model)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


HybridClassifier(
  (quantum): TorchConnector()
  (classifier): Linear(in_features=1, out_features=1, bias=True)
)


In [14]:
model_04B = create_model(
    num_qubits=4,
    seed=42
)
model = HybridClassifier(
    quantum_model=quantum_model
)

model = model.to(DEVICE)

print(model)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


HybridClassifier(
  (quantum): TorchConnector()
  (classifier): Linear(in_features=1, out_features=1, bias=True)
)


In [17]:
print(type(model_04B))
print(model_04B)

print(type(model))
print(model)
import inspect
print(inspect.getsource(type(model)))

<class 'qiskit_machine_learning.connectors.torch_connector.TorchConnector'>
TorchConnector()
<class 'src.models.hybrid_classifier.HybridClassifier'>
HybridClassifier(
  (quantum): TorchConnector()
  (classifier): Linear(in_features=1, out_features=1, bias=True)
)
class HybridClassifier(nn.Module):
    """
    Hybrid quantum-classical binary classifier.

    Parameters
    ----------
    quantum_model : nn.Module
        Quantum neural network wrapped as a PyTorch module.
    """

    def __init__(
        self,
        quantum_model,
    ):
        super().__init__()

        self.quantum = quantum_model

        self.classifier = nn.Linear(
            in_features=1,
            out_features=1,
        )

    def forward(self, x):
        """
        Forward pass.

        Parameters
        ----------
        x : torch.Tensor
            Input tensor with shape (batch_size, num_features).

        Returns
        -------
        torch.Tensor
            Binary classification logits.


In [20]:
import inspect
from src.models import quantum_model

# print(inspect.getsource(quantum_model))
print(dir(quantum_model))
print(inspect.getsource(quantum_model.create_model))

['DEFAULT_SEED', 'EstimatorQNN', 'QuantumCircuit', 'SparsePauliOp', 'StatevectorEstimator', 'TorchConnector', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'create_ansatz', 'create_feature_map', 'create_model', 'create_qnn', 'create_quantum_circuit']
def create_model(
    num_qubits=4,
    seed=DEFAULT_SEED,
):
    """
    Create a PyTorch-compatible quantum model.

    Parameters
    ----------
    num_qubits : int
        Number of qubits.

    seed : int
        Random seed for deterministic quantum evaluation.

    Returns
    -------
    TorchConnector
        PyTorch-compatible quantum neural network.
    """

    qnn = create_qnn(
        num_qubits=num_qubits,
        seed=seed,
    )

    model = TorchConnector(qnn)

    return model



In [21]:
import torch
from src.models.quantum_model import create_model
from src.models.hybrid_classifier import HybridClassifier

# Create the quantum component
quantum_model_04B = create_model(
    num_qubits=4,
    seed=42,
)

# Wrap it in the same HybridClassifier architecture
model_04B = HybridClassifier(
    quantum_model=quantum_model_04B
)

print(model_04B)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


HybridClassifier(
  (quantum): TorchConnector()
  (classifier): Linear(in_features=1, out_features=1, bias=True)
)


In [22]:
checkpoint_path = "../results/models/vqc_04B_scaled_seed42.pt"

state_dict = torch.load(
    checkpoint_path,
    map_location="cpu"
)

model_04B.load_state_dict(state_dict)

model_04B.eval()

print("04B checkpoint loaded successfully.")

04B checkpoint loaded successfully.


In [23]:
import joblib

scaler_path = "../results/preprocessing/standard_scaler_04B_seed42.joblib"

scaler_04B = joblib.load(scaler_path)

print("Scaler loaded successfully.")

Scaler loaded successfully.


In [25]:
X_test_scaled_04B = scaler_04B.transform(
    X_test
)

print("Scaled test shape:", X_test_scaled_04B.shape)

print("\nMean:")
print(X_test_scaled_04B.mean(axis=0))

print("\nStd:")
print(X_test_scaled_04B.std(axis=0))

Scaled test shape: (2956, 4)

Mean:
[0.0017654  0.00180763 0.01846003 0.03138886]

Std:
[1.00099269 1.00601958 0.99919916 0.98443775]


In [29]:
import numpy as np

model_04B.eval()

X_test_tensor = torch.tensor(
    X_test_scaled_04B,
    dtype=torch.float32
)

with torch.no_grad():
    logits = model_04B(X_test_tensor)

logits = logits.detach().cpu().numpy().reshape(-1)

predictions_reproduced = (
    logits >= 0
).astype(np.int32)

print("Logits shape:", logits.shape)
print("Predictions shape:", predictions_reproduced.shape)

saved_predictions = np.load(
    "../results/adversarial/clean_predictions_04B_seed42.npy"
)

print("Saved predictions shape:", saved_predictions.shape)

agreement = np.mean(
    predictions_reproduced == saved_predictions
)

different = np.sum(
    predictions_reproduced != saved_predictions
)

print("=" * 60)
print("04B CHECKPOINT VALIDATION")
print("=" * 60)

print(f"Prediction agreement: {agreement:.10f}")
print(f"Different predictions: {different}")

Logits shape: (2956,)
Predictions shape: (2956,)
Saved predictions shape: (2956,)
04B CHECKPOINT VALIDATION
Prediction agreement: 0.9604194858
Different predictions: 117


In [32]:
import inspect
from src.models import quantum_model

print(inspect.getsource(quantum_model.create_qnn))

print(inspect.getsource(quantum_model.create_quantum_circuit))

print(inspect.getsource(quantum_model.create_feature_map))

print(inspect.getsource(quantum_model.create_ansatz))

def create_qnn(
    num_qubits=4,
    seed=DEFAULT_SEED,
):
    """
    Create the EstimatorQNN.

    Input gradients are explicitly enabled because
    FGSM requires gradients with respect to the input
    features.
    """

    qc, feature_map, ansatz = create_quantum_circuit(
        num_qubits=num_qubits
    )

    observable = SparsePauliOp.from_list(
        [
            ("ZIII", 1.0)
        ]
    )

    estimator = StatevectorEstimator(
        seed=seed
    )

    qnn = EstimatorQNN(
        circuit=qc,
        estimator=estimator,
        observables=observable,
        input_params=feature_map.parameters,
        weight_params=ansatz.parameters,
        input_gradients=True,
    )

    return qnn

def create_quantum_circuit(
    num_qubits=4,
    feature_reps=2,
    ansatz_reps=2,
):
    """
    Create the complete variational quantum circuit.

    Parameters
    ----------
    num_qubits : int
        Number of qubits.

    feature_reps : int
        Number of repetitions 

In [33]:
import json

reference_path = "../results/adversarial/clean_reference_04B_seed42.json"

with open(reference_path, "r") as f:
    reference = json.load(f)

print(json.dumps(reference, indent=2))

{
  "experiment": "04B_clean_reference",
  "seed": 42,
  "num_qubits": 4,
  "accuracy": 0.5852503382949933,
  "precision": 0.5915049816465653,
  "recall": 0.7161904761904762,
  "f1": 0.6479035037334865,
  "clean_correct_samples": 1730,
  "total_test_samples": 2956
}


In [35]:
print("Checkpoint parameters:")
for name, param in model_04B.named_parameters():
    print(name, param.detach().cpu().numpy())

print("\nQuantum weights:")
print(model_04B.quantum.weight.detach().cpu().numpy())

print("\nInternal quantum weights:")
print(model_04B.quantum._weights.detach().cpu().numpy())

print("\nClassifier weight:")
print(model_04B.classifier.weight.detach().cpu().numpy())

print("\nClassifier bias:")
print(model_04B.classifier.bias.detach().cpu().numpy())

Checkpoint parameters:
quantum.weight [ 1.5234425   1.3247753  -0.30822906  1.6413008  -1.8245659   0.00345224
 -0.04901591  1.5064764   0.8815429  -0.73362815  0.8691962   0.1010682 ]
classifier.weight [[2.050361]]
classifier.bias [0.04361313]

Quantum weights:
[ 1.5234425   1.3247753  -0.30822906  1.6413008  -1.8245659   0.00345224
 -0.04901591  1.5064764   0.8815429  -0.73362815  0.8691962   0.1010682 ]

Internal quantum weights:
[ 1.5234425   1.3247753  -0.30822906  1.6413008  -1.8245659   0.00345224
 -0.04901591  1.5064764   0.8815429  -0.73362815  0.8691962   0.1010682 ]

Classifier weight:
[[2.050361]]

Classifier bias:
[0.04361313]


In [38]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
print("Saved accuracy:",
      accuracy_score(y_test, saved_predictions))

print("Reconstructed accuracy:",
      accuracy_score(y_test, predictions_reproduced))

print("Saved prediction distribution:")
print(np.unique(saved_predictions, return_counts=True))

print("\nReconstructed prediction distribution:")
print(np.unique(predictions_reproduced, return_counts=True))

different_mask = (
    predictions_reproduced != saved_predictions
)

print("Different samples:", different_mask.sum())

print("\nReconstructed logits for different samples:")
print(logits[different_mask][:30])

print("\nNumber close to decision boundary:")
print(
    np.sum(
        np.abs(logits[different_mask]) < 0.1
    )
)

Saved accuracy: 0.5852503382949933
Reconstructed accuracy: 0.5815290933694182
Saved prediction distribution:
(array([0, 1], dtype=int32), array([1049, 1907]))

Reconstructed prediction distribution:
(array([0, 1], dtype=int32), array([1054, 1902]))
Different samples: 117

Reconstructed logits for different samples:
[ 0.00888946  0.01632049 -0.04067612  0.035224   -0.02966863  0.0152795
 -0.10275826 -0.00628433  0.0021933  -0.01072632  0.02937801  0.01564564
 -0.00731504  0.0370874  -0.10102788 -0.00057038  0.01517936  0.00657321
 -0.0094136  -0.00563277 -0.01092564  0.03084595  0.05268818 -0.03634557
 -0.00352557  0.02531523 -0.0076958   0.09600237 -0.02731145 -0.02653887]

Number close to decision boundary:
114


In [39]:
def get_predictions(model, X):
    model.eval()

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32
    )

    with torch.no_grad():
        logits = model(X_tensor)

    logits = logits.detach().cpu().numpy().reshape(-1)

    predictions = (
        logits >= 0
    ).astype(np.int32)

    return predictions, logits


pred1, logits1 = get_predictions(
    model_04B,
    X_test_scaled_04B
)

pred2, logits2 = get_predictions(
    model_04B,
    X_test_scaled_04B
)

print(
    "Prediction agreement between repeated evaluations:",
    np.mean(pred1 == pred2)
)

print(
    "Maximum logit difference:",
    np.max(np.abs(logits1 - logits2))
)

print(
    "Number of prediction differences:",
    np.sum(pred1 != pred2)
)

Prediction agreement between repeated evaluations: 1.0
Maximum logit difference: 0.0
Number of prediction differences: 0


In [28]:
agreement = np.mean(
    predictions_reproduced == saved_predictions
)

different = np.sum(
    predictions_reproduced != saved_predictions
)

print("=" * 60)
print("04B CHECKPOINT VALIDATION")
print("=" * 60)

print(f"Prediction agreement: {agreement:.10f}")
print(f"Different predictions: {different}")

04B CHECKPOINT VALIDATION
Prediction agreement: 0.9604194858
Different predictions: 117


In [15]:
import torch

checkpoint_path = "../results/models/vqc_04B_scaled_seed42.pt"

state_dict = torch.load(
    checkpoint_path,
    map_location="cpu"
)

model_04B.load_state_dict(state_dict)

model_04B.eval()

print(model_04B)
print("04B checkpoint loaded successfully.")

RuntimeError: Error(s) in loading state_dict for TorchConnector:
	Missing key(s) in state_dict: "weight", "_weights". 
	Unexpected key(s) in state_dict: "quantum.weight", "quantum._weights", "classifier.weight", "classifier.bias". 

In [14]:
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

In [15]:
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=DEVICE,
)

In [16]:
import time
import torch

print("Model:")
print(model)

print("\nDataset:")
print("X_train:", X_train_scaled.shape)
print("X_val:", X_val_scaled.shape)
print("X_test:", X_test_scaled.shape)

print("\nDevice:")
print(torch.device("cuda" if torch.cuda.is_available() else "cpu"))

Model:
HybridClassifier(
  (quantum): TorchConnector()
  (classifier): Linear(in_features=1, out_features=1, bias=True)
)

Dataset:
X_train: (9459, 4)
X_val: (2365, 4)
X_test: (2956, 4)

Device:
cpu


In [17]:
print("Batch size:", BATCH_SIZE)
print("Epochs:", EPOCHS)
print("Training samples:", len(X_train_scaled))

Batch size: 32
Epochs: 30
Training samples: 9459


In [21]:
import pandas as pd
import json
import os

fgsm_csv = "../results/adversarial/fgsm_04B_seed42.csv"
fgsm_json = "../results/adversarial/fgsm_04B_seed42.json"

print("CSV exists:", os.path.exists(fgsm_csv))
print("JSON exists:", os.path.exists(fgsm_json))

if os.path.exists(fgsm_csv):
    fgsm_results = pd.read_csv(fgsm_csv)
    
    print("\nFGSM RESULTS")
    print("=" * 70)
    print(fgsm_results.to_string(index=False))

if os.path.exists(fgsm_json):
    print("\nFGSM JSON")
    print("=" * 70)
    
    with open(fgsm_json, "r") as f:
        fgsm_metadata = json.load(f)
    
    print(json.dumps(fgsm_metadata, indent=2))

metadata_path = "../results/models/vqc_04B_scaled_seed42_metadata.json"

with open(metadata_path, "r") as f:
    metadata = json.load(f)

print(json.dumps(metadata, indent=2))

CSV exists: True
JSON exists: True

FGSM RESULTS
 epsilon  clean_accuracy  clean_precision  clean_recall  clean_f1  adversarial_accuracy  adversarial_precision  adversarial_recall  adversarial_f1  accuracy_drop  prediction_change_rate  attack_success_rate  clean_correct_samples  successful_attacks  mean_l2_perturbation  max_l2_perturbation  mean_linf_perturbation  max_linf_perturbation  runtime_seconds
    0.01         0.58525         0.591505       0.71619  0.647904              0.489851               0.517943            0.613968        0.561883       0.095399                0.102165             0.168786                   1730                 292                  0.02                 0.02                    0.01                   0.01       476.061335
    0.05         0.58525         0.591505       0.71619  0.647904              0.340663               0.387077            0.406984        0.396781       0.244587                0.312246             0.475723                   1730        

In [23]:
import torch
import os

checkpoint_path = "../results/models/vqc_04B_scaled_seed42.pt"

print("Checkpoint exists:", os.path.exists(checkpoint_path))

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu"
)

print(type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    print(checkpoint.keys())

if isinstance(checkpoint, dict):
    for key, value in checkpoint.items():
        if hasattr(value, "shape"):
            print(key, value.shape)
        else:
            print(key, type(value))

Checkpoint exists: True
<class 'collections.OrderedDict'>

Checkpoint keys:
odict_keys(['quantum.weight', 'quantum._weights', 'classifier.weight', 'classifier.bias'])
quantum.weight torch.Size([12])
quantum._weights torch.Size([12])
classifier.weight torch.Size([1, 1])
classifier.bias torch.Size([1])


In [24]:
clean_pred_path = "../results/adversarial/clean_predictions_04B_seed42.npy"

clean_predictions_saved = np.load(clean_pred_path)

print("Shape:", clean_predictions_saved.shape)
print("Unique predictions:", np.unique(clean_predictions_saved, return_counts=True))

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("Accuracy:",
      accuracy_score(y_test, clean_predictions_saved))

print("Precision:",
      precision_score(y_test, clean_predictions_saved, zero_division=0))

print("Recall:",
      recall_score(y_test, clean_predictions_saved, zero_division=0))

print("F1:",
      f1_score(y_test, clean_predictions_saved, zero_division=0))

Shape: (2956,)
Unique predictions: (array([0, 1], dtype=int32), array([1049, 1907]))
Accuracy: 0.5852503382949933
Precision: 0.5915049816465653
Recall: 0.7161904761904762
F1: 0.6479035037334865


In [17]:
import time

start_time = time.perf_counter()

history_04B = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
)

elapsed = time.perf_counter() - start_time

print()
print("=" * 50)
print("04B TRAINING COMPLETE")
print("=" * 50)
print(f"Training time: {elapsed / 60:.2f} minutes")

Epoch 001/030 | Train Loss: 0.6740 | Val Loss: 0.6699 | Train Acc: 0.5979 | Val Acc: 0.5987


KeyboardInterrupt: 

In [ ]:
test_loss_04B, test_metrics_04B = trainer.evaluate(
    test_loader
)

print("=" * 50)
print("04B — SCALED CLEAN VQC")
print("=" * 50)

print(f"Test loss      : {test_loss_04B:.4f}")
print(f"Test accuracy  : {test_metrics_04B['accuracy']:.4f}")
print(f"Test precision : {test_metrics_04B['precision']:.4f}")
print(f"Test recall    : {test_metrics_04B['recall']:.4f}")
print(f"Test F1        : {test_metrics_04B['f1']:.4f}")

04B — SCALED CLEAN VQC
Test loss      : 0.6774
Test accuracy  : 0.5805
Test precision : 0.5873
Test recall    : 0.7156
Test F1        : 0.6451


In [ ]:
MODEL_DIR = PROJECT_ROOT / "results" / "models"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_04B_PATH = (
    MODEL_DIR
    / "vqc_04B_scaled_seed42.pt"
)

torch.save(
    model.state_dict(),
    MODEL_04B_PATH
)

print("=" * 50)
print("MODEL CHECKPOINT SAVED")
print("=" * 50)

print(MODEL_04B_PATH)

MODEL CHECKPOINT SAVED
c:\Work\Quantum-Adversarial-Robustness\results\models\vqc_04B_scaled_seed42.pt


In [ ]:
import json

metadata = {
    "experiment": "04B",
    "description": "Clean VQC with standardized PCA features",

    "seed": SEED,

    "num_qubits": NUM_QUBITS,

    "feature_dimension": 4,

    "feature_map": {
        "type": "ZZFeatureMap",
        "reps": 2
    },

    "ansatz": {
        "type": "RealAmplitudes",
        "reps": 2,
        "entanglement": "linear"
    },

    "gradient": "default Qiskit gradient / parameter-shift",

    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,

    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,

    "scaling": "StandardScaler",

    "test_metrics": {
        "loss": float(test_loss_04B),
        "accuracy": float(test_metrics_04B["accuracy"]),
        "precision": float(test_metrics_04B["precision"]),
        "recall": float(test_metrics_04B["recall"]),
        "f1": float(test_metrics_04B["f1"]),
    }
}

METADATA_PATH = (
    MODEL_DIR
    / "vqc_04B_scaled_seed42_metadata.json"
)

with open(
    METADATA_PATH,
    "w"
) as f:
    json.dump(
        metadata,
        f,
        indent=4
    )

print("Metadata saved:")
print(METADATA_PATH)

Metadata saved:
c:\Work\Quantum-Adversarial-Robustness\results\models\vqc_04B_scaled_seed42_metadata.json
